# 06 Revenue from able-to-pay non-payers

How much could households that can afford a bill but do not pay add each year? Scenarios, not forecasts.

In [1]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parents[2]      # repo root, two levels above this folder
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import pandas as pd

from src.common.config import load_config
from src.common.pipeline import build

cfg = load_config()
MODE = os.environ.get("DATA_MODE")          # None = use config; "synthetic" = fake
ghs, ies, nfcm = build(MODE, save=False)

C:\Users\mmjmo\Projects\sa-electricity-nonpayment\src\common\pipeline.py:28: UnicodeWarning: 
One or more strings in the dta file could not be decoded using utf-8, and
so the fallback encoding of latin-1 is being used.  This can happen when a file
has been incorrectly encoded by Stata or some other software. You should verify
the string values returned are correct.
  return pd.read_stata(full, convert_categoricals=False)


**Note** 1.8 million able households at an average bill of R3,264 a year gives R5.9 billion at survey prices. Multiply by the 1.34 price factor to move from 2022/23 to 2025/26 prices: about R7.9 billion.

In [2]:
print(round(1.8e6 * 3264 / 1e9, 2), round(1.8e6 * 3264 * 1.34 / 1e9, 2))   # 5.88 7.88

5.88 7.87


In [3]:
from src.a_who_pays import revenue

rev = revenue.revenue_table(ies, ghs, cfg)
headline = rev[(rev["threshold"] == cfg["thresholds"]["headline"]) & (rev["collection_rate"] == 1.0)]
headline[["pool_share", "able_households_m", "avg_bill_rand_survey", "revenue_rbn_low",
          "revenue_rbn_central", "revenue_rbn_high", "eskom_direct_rbn_central",
          "pct_of_allowable_revenue"]].round(2)

,pool_share,able_households_m,avg_bill_rand_survey,revenue_rbn_low,revenue_rbn_central,revenue_rbn_high,eskom_direct_rbn_central,pct_of_allowable_revenue
14,0.3,1.56,2512.03,4.98,5.25,5.91,2.71,1.37
17,0.5,1.80,3264.33,7.47,7.88,8.86,4.06,2.05
20,0.7,1.98,4207.59,10.57,11.14,12.54,5.74,2.90
23,1.0,2.17,7363.80,20.28,21.38,24.06,11.02,5.56


### How much revenue could able-to-pay non-payers add?
At the 10% line, with full collection and 2025/26 prices, the answer depends on who the non-payers are. If they resemble the poorest 30% of paying households it is about R5.3 billion a year, for the poorest 50% about R7.9 billion, for the poorest 70% about R11.1 billion, and if they resemble a typical payer about R21.4 billion. The price update adds a range of roughly 5% below to 12% above the central figure. I treat R5 to R11 billion as the realistic range. The typical-payer case is probably too high, because it uses an average bill of R7,364, while the poorest-50% case uses R3,264, and non-payers are likely to use less electricity than the average payer.

In [4]:
one = rev[(rev["threshold"] == cfg["thresholds"]["headline"]) & (rev["pool_share"] == 0.5)]
one[["collection_rate", "paying_households_m", "revenue_rbn_central", "eskom_direct_rbn_central"]].round(2)

,collection_rate,paying_households_m,revenue_rbn_central,eskom_direct_rbn_central
15,0.5,0.90,3.94,2.03
16,0.7,1.26,5.51,2.84
17,1.0,1.80,7.88,4.06


### How much would reach Eskom, and how big is that?
About 51.5% of non-paying households are supplied by Eskom and 44% by municipalities, so about half of the extra revenue would go to Eskom directly: about R2.7 to R5.7 billion a year in the realistic cases. That is about 1.4% to 2.9% of the R384.61 billion allowable revenue, or roughly 3 to 6 c/kWh if it were all passed through, using the placeholder sales volume. Money paid to a municipality only helps Eskom if the municipality then pays its Eskom bill. If only 70% of the able households were collected from, the 50% case falls from R7.9 billion to R5.5 billion, and the Eskom share from R4.1 billion to R2.8 billion.

In [5]:
revenue.bounds_table(ies, ghs, cfg).round(2)

,threshold,assumption,able_households_m,revenue_rbn_central
0,0.08,poorest_first,1.07,3.38
1,0.08,richest_first,2.57,47.25
2,0.10,poorest_first,1.35,4.62
3,0.10,richest_first,2.63,49.75
4,0.12,poorest_first,1.57,5.68
5,0.12,richest_first,2.65,50.84


In [6]:
revenue.supplier_shares(ghs)   # who supplies the non-paying households

{'eskom': 0.5152910046666777,
 'municipality': 0.44004009501817315,
 'other': 0.04466890031514915}

### Limits 

1. I do not know who the non-payers are, so the result depends on the pool assumption. The two extreme cases (poorest-first about R4.6 billion, richest-first about R50 billion) show how wide the possibilities are. The richest-first case is not realistic.
2. Able to pay does not mean willing to pay, and collection would not reach 100%.
3. The bills I use are those of households that pay. I do not know what a non-payer would use.
4. Bills are scaled from 2022/23 to 2025/26 using tariff increases (factors 1.27 to 1.51), assuming consumption stays the same.
5. I assume Eskom's share of the extra revenue equals its share of non-paying households (51.5%). I have not netted off free basic electricity, collection costs, or the sales volume, which is still a placeholder.